# Emergency Department Analytics: From Statistics to Predictive Models

## Overview

Emergency departments handle thousands of patient visits a year, yet key decisions — who gets admitted, how long patients stay, which triage levels drive congestion — are often made without data-driven support. This notebook builds a complete analytics pipeline to change that.

**Dataset:** 400 ED patient visits with demographics, triage level, length of stay, diagnostic test counts, and admission outcome.

**What this notebook covers:**
1. Dataset inspection and structure
2. Length of stay summary statistics
3. LOS comparison by admission status (visualization)
4. Hypothesis testing — t-test
5. ANOVA across triage levels
6. Logistic regression admission model
7. ROC-AUC evaluation
8. Decision tree classifier
9. Random forest classifier
10. Operational interpretation


---
## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
print("Libraries loaded")

## Load Dataset

In [ ]:
df = pd.read_csv("data/ed_visits.csv")
df.head()

---
## 1. Dataset Inspection

Before any analysis, I check the shape of the dataset and the data type of every column — this determines which statistical methods are appropriate at each step.

In [ ]:
# Shape: rows and columns
print(f"Rows: {df.shape[0]}  |  Columns: {df.shape[1]}\n")

# Data type of each variable
print("Column data types:")
print(df.dtypes.to_string())

---
## 2. ED Length of Stay — Summary Statistics

Computing mean and median gives us the central tendency of LOS. The gap between the two reveals whether the distribution is skewed.

In [ ]:
mean_los   = df['ed_los_hours'].mean()
median_los = df['ed_los_hours'].median()

print(f"Mean LOS:   {mean_los:.2f} hours")
print(f"Median LOS: {median_los:.2f} hours")
print(f"\nThe mean ({mean_los:.2f}) > median ({median_los:.2f}) indicates right skew —")
print("a minority of long-stay patients pull the average upward.")

---
## 3. LOS by Admission Status — Visualization

A boxplot lets us visually compare the LOS distributions for admitted vs discharged patients before running any formal statistical test.

In [ ]:
df['Admission Status'] = df['admitted'].map({1: 'Admitted', 0: 'Discharged'})

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Boxplot: admission status
sns.boxplot(data=df, x='Admission Status', y='ed_los_hours',
            palette={'Admitted': '#E05C5C', 'Discharged': '#5C9BE0'}, ax=axes[0])
axes[0].set_title('LOS by Admission Status', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Length of Stay (hours)')

# Boxplot: triage level
sns.boxplot(data=df, x='triage_level', y='ed_los_hours',
            order=['Low','Medium','High'],
            palette={'Low':'#5CB85C','Medium':'#F0AD4E','High':'#D9534F'}, ax=axes[1])
axes[1].set_title('LOS by Triage Level', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Length of Stay (hours)')

plt.suptitle('Emergency Department — Length of Stay Comparison', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/los_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Hypothesis Test — Admitted vs Not Admitted

An independent samples t-test checks whether the difference in mean LOS between the two groups is statistically significant or could be explained by chance.

In [ ]:
admitted     = df[df['admitted'] == 1]['ed_los_hours']
not_admitted = df[df['admitted'] == 0]['ed_los_hours']

print(f"Admitted     — mean: {admitted.mean():.2f} hrs  (n={len(admitted)})")
print(f"Not admitted — mean: {not_admitted.mean():.2f} hrs  (n={len(not_admitted)})")

t_stat, p_val = stats.ttest_ind(admitted, not_admitted)
print(f"\nT-statistic: {t_stat:.4f}")
print(f"P-value:     {p_val:.6f}")
print(f"\n{'Significant' if p_val < 0.05 else 'Not significant'} at α = 0.05")

---
## 5. ANOVA — LOS across Triage Levels

With three triage categories (Low / Medium / High), ANOVA tests whether any group mean differs significantly from the others.

In [ ]:
for level in ['Low','Medium','High']:
    grp = df[df['triage_level'] == level]['ed_los_hours']
    print(f"{level:<8} — mean: {grp.mean():.2f} hrs  (n={len(grp)})")

groups  = [g['ed_los_hours'].values for _, g in df.groupby('triage_level')]
f_stat, p_anova = stats.f_oneway(*groups)

print(f"\nF-statistic: {f_stat:.4f}")
print(f"P-value:     {p_anova:.6f}")
print(f"\n{'Significant differences exist across triage levels.' if p_anova < 0.05 else 'No significant difference.'}")

---
## 6. Logistic Regression — Admission Risk Model

Logistic regression predicts the probability of admission using three early-available signals: age, ED length of stay, and number of tests ordered.

In [ ]:
X = df[['age', 'ed_los_hours', 'num_tests_ordered']]
y = df['admitted']

logistic_model = LogisticRegression(max_iter=1000, random_state=42)
logistic_model.fit(X, y)

print(f"Logistic Regression Accuracy: {logistic_model.score(X, y):.4f}")
print("\nCoefficients:")
for feat, coef in zip(X.columns, logistic_model.coef_[0]):
    print(f"  {feat:<22} {coef:+.4f}")

---
## 7. ROC-AUC Evaluation

ROC-AUC measures how well the model separates admitted from discharged patients across all possible decision thresholds. A value of 1.0 is perfect; 0.5 is random.

In [ ]:
probs = logistic_model.predict_proba(X)[:, 1]
auc   = roc_auc_score(y, probs)

print(f"ROC-AUC: {auc:.4f}")
print(f"\nThe model correctly ranks a randomly selected admitted patient")
print(f"above a discharged one {auc:.1%} of the time.")

---
## 8. Decision Tree Classifier

A decision tree learns explicit if-then rules from the data. It is highly interpretable but prone to overfitting when no depth constraint is applied.

In [ ]:
decision_tree_model = DecisionTreeClassifier(random_state=42)
decision_tree_model.fit(X, y)

dt_acc = decision_tree_model.score(X, y)
print(f"Decision Tree Accuracy: {dt_acc:.4f}")
if dt_acc == 1.0:
    print("\nNote: Perfect training accuracy indicates the tree has memorized the")
    print("training data. A train/test split is needed for a fair performance estimate.")

---
## 9. Random Forest Classifier

A random forest builds many decision trees on random subsets of data and averages their predictions — reducing variance compared to a single tree.

In [ ]:
random_forest_model = RandomForestClassifier(n_estimators=100, random_state=42)
random_forest_model.fit(X, y)

rf_acc = random_forest_model.score(X, y)
print(f"Random Forest Accuracy: {rf_acc:.4f}")

print("\nFeature importances:")
for feat, imp in sorted(zip(X.columns, random_forest_model.feature_importances_), key=lambda x: -x[1]):
    print(f"  {feat:<22} {imp:.4f}")

---
## 10. Operational Interpretation

Based on the full analysis, which statement best describes the findings?

In [ ]:
# Answer: A
#
# Patients with longer ED length of stay and a higher number of diagnostic
# tests have an increased likelihood of hospital admission, suggesting these
# variables can be used for early risk stratification.
#
# Supporting evidence:
#   - t-test p < 0.0001: admitted patients stay significantly longer
#   - ANOVA p < 0.0001: triage level strongly associated with LOS
#   - Logistic regression AUC = 0.776: good discriminatory ability
#   - Random forest confirms ed_los_hours and num_tests as top predictors

print("Answer: A")

---
## Summary

| Analysis | Result |
|---|---|
| Mean LOS | 5.54 hours |
| Median LOS | 4.49 hours |
| T-test (admitted vs discharged) | T=8.87, p < 0.0001 ✓ |
| ANOVA (triage levels) | F=56.95, p < 0.0001 ✓ |
| Logistic Regression Accuracy | 70.5% |
| ROC-AUC | 0.776 |
| Decision Tree Accuracy | 100% (overfitting) |
| Random Forest Accuracy | 100% (overfitting) |

**Operational conclusion:** LOS and number of tests ordered are the strongest early predictors of admission. Deploying a real-time scoring model using these variables could enable earlier bed requests and better patient flow management across the ED.
